In [ ]:
import sys, os, shutil, tempfile, warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._ts_dataloader      import DataLoaderFactory
from dataloaders.ts_sharding        import (
    write_sharded_dataset,
    ShardedTrainDataset,
    ShardedValDataset,
    ShardedTestDataset,
)
from common.train import train, eval_test
from models.linear import TinyLinearModel
from losses.torch_losses import get_loss
from common.ensembling import Ensembler

warnings.filterwarnings("ignore")
print("imports OK")


In [ ]:
# ── Synthetic dataframe factory ───────────────────────────────────────────────

def make_df(
    n_series: int = 3,
    n_steps:  int = 500,
    n_hist:   int = 1,       # number of hist exog cols
    seed:     int = 42,
) -> pd.DataFrame:
    """
    Returns a long-format DataFrame with columns:
        unique_id, ds, y, available_mask, hist_0, hist_1, ...
    All series have equal length (n_steps timesteps).
    available_mask is 1 everywhere (no gaps).
    """
    rng   = np.random.default_rng(seed)
    times = pd.date_range("2020-01-01", periods=n_steps, freq="5min")
    rows  = []
    for s in range(n_series):
        y    = rng.standard_normal(n_steps).astype(np.float32)
        hist = {f"hist_{i}": rng.standard_normal(n_steps).astype(np.float32)
                for i in range(n_hist)}
        df_s = pd.DataFrame({"unique_id": f"series_{s}", "ds": times, "y": y,
                              "available_mask": np.ones(n_steps, dtype=np.float32),
                              **hist})
        rows.append(df_s)
    return pd.concat(rows, ignore_index=True)


# ── Config factories ──────────────────────────────────────────────────────────

def make_mcfg(
    horizon:           int  = 6,
    n_channels:        int  = 3, 
    context_length:    int  = 64,
    fcd_samples:       int  = 4,
    batch_size:        int  = 2,
    max_steps:         int  = 20,
    val_check_interval:int  = 10,
    mixing_strategy:   str  = "concat",
    checkpoint_dir:    str  = "/tmp/ckpts",
    num_workers:       int  = 0,
):
    return SimpleNamespace(
        horizon                = horizon,
        n_channels             = n_channels, 
        context_length         = context_length,
        fcd_samples            = fcd_samples,
        batch_size             = batch_size,
        valid_batch_size       = batch_size,
        max_steps              = max_steps,
        val_check_interval     = val_check_interval,
        learning_rate          = 1e-3,
        gradient_clip_val      = 1.0,
        early_stopping_patience= 9999,  # disable for tests
        monitor_metric         = "loss",
        monitor_mode           = "min",
        mixing_strategy        = mixing_strategy,
        drop_last              = False,
        batch_mode             = "full_series",
        checkpoint_dir         = checkpoint_dir,
        checkpoint_step        = 99999,  # suppress mid-run saves
        num_workers            = num_workers,
        loss                   = 'mae', 
    )


def make_entry(
    path:            str,
    name:            str  = "ds",
    horizon:         int  = 6,
    val_size:        int  = 50,
    test_size:       int  = 50,
    weight:          float= 1.0,
    hist_exog_cols:  list = None,
    futr_exog_cols:  list = None,
    stat_exog_cols:  list = None,
    per_series_split:bool = False,
    use_context_head:bool = True,
    sharded_dir:     str  = None,
):
    return SimpleNamespace(
        path             = path,
        name             = name,
        horizon          = horizon,
        val_size         = val_size,
        test_size        = test_size,
        weight           = weight,
        hist_exog_cols   = hist_exog_cols  or [],
        futr_exog_cols   = futr_exog_cols  or [],
        stat_exog_cols   = stat_exog_cols  or [],
        per_series_split = per_series_split,
        use_context_head = use_context_head,
        sharded_dir      = sharded_dir,
    )


def make_dcfg(train_entries, val_entries=None, test_entries=None):
    return SimpleNamespace(
        train      = train_entries,
        validation = val_entries  or train_entries,
        test       = test_entries or train_entries,
    )


print("helpers OK")

In [ ]:
class Ensembler:
    def __init__(self, ensemble_method='mean', **kwargs):
        self.kwargs = kwargs
        methods = {
            'mean': self._cumulative_mean,
            'median': self._cumulative_median,
            'ewm': self._cumulative_ewm,
        }
        if ensemble_method not in methods:
            raise ValueError(f"ensemble_method must be one of {list(methods.keys())}")
        self.ensembler = methods[ensemble_method]

    def ensemble(self, preds, mask=None):
        """
        preds: (B, T, H, C)
        mask:  (B, T, H, C) int — 1=valid, 0=mask out
        returns: (B, T, H, C)
        """
        B, T, H, C = preds.shape
        masked_preds  = self.preds_reshape_for_ensembling(preds, mask) # (B, S, H, C)

        # reshape to (B*S, H, C) so ensemblers stay unchanged (axis=1 is overlaps)
        B_, test_size, H_, C_ = masked_preds.shape
        flat = masked_preds.reshape(B_ * test_size, H_, C_) # (B*S, H, C)
        ensembled = self.ensembler(flat, **self.kwargs) # (B*S, H, C)
        ensembled_preds = ensembled.reshape(B_, test_size, H_, C_) # (B, test_size, H, C)

        return self.ensembled_preds_reshape_for_windows(ensembled_preds) # (B, T, H, C)

    def preds_reshape_for_ensembling(self, preds, mask=None):
        B, T, H, C = preds.shape

        # Set masked predictions to nan
        if mask is not None:
            preds = np.where(mask == 0, np.nan, preds.copy()) # (B, T, H, C)

        flatten_preds = preds.reshape(B, T * H, C) # (B, T*H, C)

        # index grid — same for all batches
        idx = np.arange(T * H).reshape(T, H)
        flipped_idx = np.fliplr(idx)
        zs = np.full((H - 1, H), np.nan)
        padded_idx = np.concatenate([zs, flipped_idx, zs]) # (T + 2*(H-1), H)

        idx_windows = np.lib.stride_tricks.sliding_window_view(padded_idx, window_shape=(H, H))
        date_idx = np.diagonal(idx_windows[:, 0], axis1=1, axis2=2) # (test_size, H)

        nan_mask = ~np.isnan(date_idx) # (S, H)
        idx2 = np.where(np.isnan(date_idx), 0, date_idx).astype(int) # (S, H)

        indexed_preds = flatten_preds[:, idx2, :] # (B, S, H, C)
        nan_mask_exp = nan_mask[None, :, :, None] # (1, S, H, 1)
        return np.where(~nan_mask_exp, np.nan, indexed_preds) # (B, S, H, C)

    def ensembled_preds_reshape_for_windows(self, ensembled_preds):
        B, S, H, C = ensembled_preds.shape

        # flip H axis so diagonals align — same as before but over batch
        flipped = np.flip(ensembled_preds, axis=2) # (B, S, H, C)

        windows = np.lib.stride_tricks.sliding_window_view(flipped, window_shape=(H, H), axis=(1, 2))
        return np.diagonal(windows[:, :, 0], axis1=2, axis2=3) # (B, n_windows, H, C)

    def _cumulative_mean(self, preds, window_size=None, **kwargs):
        _, H, _ = preds.shape
        if window_size is None:
            cumsum = np.nancumsum(preds, axis=1)
            counts = np.cumsum(~np.isnan(preds), axis=1)
        else:
            cumsum = np.stack([np.nansum(preds[:, max(0, h-window_size+1):h+1, :], axis=1) for h in range(H)], axis=1)
            counts = np.stack([np.sum(~np.isnan(preds[:, max(0, h-window_size+1):h+1, :]), axis=1) for h in range(H)], axis=1)
        ensemble = cumsum / counts
        ensemble[counts == 0] = np.nan
        return ensemble # (B, H, C)

    def _cumulative_median(self, preds, window_size=None, **kwargs):
        _, H, _ = preds.shape
        return np.stack([
            np.nanmedian(preds[:, max(0, h-window_size+1) if window_size else 0:h+1, :], axis=1)
            for h in range(H)
        ], axis=1) # (B, H, C)

    def _cumulative_ewm(self, preds, alpha=0.5, window_size=None, **kwargs):
        B, H, C = preds.shape
        beta = np.log(alpha / (1 - alpha))
        W = window_size if window_size is not None else H

        h_idx = np.arange(H)
        i_idx = np.arange(H)
        starts = np.maximum(0, h_idx - W + 1)

        mask = (i_idx[None, :] >= starts[:, None]) & (i_idx[None, :] <= h_idx[:, None])
        weight_matrix = np.where(mask, np.exp(beta * (i_idx[None, :] - starts[:, None])), 0.0)

        valid = ~np.isnan(preds)
        preds_clean = np.nan_to_num(preds)
        weighted_sum = np.einsum('hi,bic->bhc', weight_matrix, valid * preds_clean)
        w_sum = np.einsum('hi,bic->bhc', weight_matrix, valid.astype(float))

        return np.where(w_sum > 0, weighted_sum / w_sum, np.nan) # (B, H, C)


In [ ]:
print("=" * 60)
print("TEST — single dataset, concat mixing")
print("=" * 60)

with tempfile.TemporaryDirectory() as tmpdir:
    csv_path = f"{tmpdir}/data.csv"
    df = make_df(n_series=3, n_steps=300)
    df.to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        context_length = 32,
        fcd_samples    = 4,
        batch_size     = 2,
        max_steps      = 20,
        mixing_strategy= "concat",
        checkpoint_dir = tmpdir,
    )
    entry = make_entry(csv_path, name="test", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    model   = TinyLinearModel(mcfg)
    metrics = train(model, mcfg, train_loader, val_loaders,
                    device=torch.device("cpu"), seed=0)

    print(f"\nfinal val metrics: {metrics}")
    assert Path(tmpdir, "final.pt").exists(), "final.pt not saved"

    # Check predictions
    test_loaders = factory.test_dataloaders()
    preds = eval_test(model, factory, device=torch.device("cpu"))
    assert "test" in preds, "missing test predictions"



print("\n✓ TEST PASSED")

In [ ]:
preds2 = preds['test']['preds']
preds2 = preds2.unsqueeze(0)
preds2 = preds2.detach().cpu().numpy()

In [ ]:
ensembler = Ensembler(ensemble_method='mean')
ensembled_windows = ensembler.ensemble(preds2)

In [ ]:
ensembled_windows